# 🤖 AI Agent Excel Assistant — Experimentation Notebook

This notebook demonstrates the full workflow: loading data, defining tools,
building the agent, and testing different scenarios.

## Setup
Make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. Configured `.env` with at least one LLM API key

In [1]:
import sys
sys.path.insert(0, '..')

# Verify imports
from app.config import settings
print(f"✅ Config loaded — Provider: {settings.llm_provider.value}, Model: {settings.active_model}")

✅ Config loaded — Provider: groq, Model: llama-3.3-70b-versatile


## 1. Loading and Exploring Data

The DataManager loads both Excel files into pandas DataFrames on initialization.

In [2]:
from app.data.manager import DataManager

dm = DataManager()

# List all available datasets
for ds in dm.list_datasets():
    print(f"📊 {ds['display_name']}: {ds['rows']} rows, {len(ds['columns'])} columns")
    print(f"   Columns: {ds['columns']}")
    print()

📊 Real Estate Listings: 1000 rows, 11 columns
   Columns: ['Listing ID', 'Property Type', 'City', 'State', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Year Built', 'List Price', 'Sale Price', 'Listing Status']

📊 Marketing Campaigns: 1000 rows, 11 columns
   Columns: ['Campaign ID', 'Campaign Name', 'Channel', 'Start Date', 'End Date', 'Budget Allocated', 'Amount Spent', 'Impressions', 'Clicks', 'Conversions', 'Revenue Generated']



In [3]:
# Inspect the Real Estate Listings schema
import json

schema = dm.get_schema("real_estate_listings")
print(f"Dataset: {schema['display_name']}")
print(f"Total rows: {schema['total_rows']}")
print(f"ID column: {schema['id_column']}")
print("\nColumns:")
for col in schema['columns']:
    info = f"  {col['name']} ({col['type']})"
    if 'unique_values' in col:
        info += f" — values: {col['unique_values']}"
    elif 'min' in col:
        info += f" — range: {col['min']:.0f} to {col['max']:.0f}, avg: {col['mean']:.0f}"
    print(info)

Dataset: Real Estate Listings
Total rows: 1000
ID column: Listing ID

Columns:
  Listing ID (text)
  Property Type (text) — values: ['Apartment', 'Condo', 'House', 'Townhouse']
  City (text)
  State (text) — values: ['Arizona', 'California', 'Colorado', 'Florida', 'Georgia', 'Illinois', 'Massachusetts', 'New York', 'Texas', 'Washington']
  Bedrooms (integer) — range: 1 to 5, avg: 3
  Bathrooms (decimal) — range: 1 to 4, avg: 2
  Square Footage (integer) — range: 240 to 2970, avg: 1107
  Year Built (integer) — range: 1960 to 2025, avg: 1992
  List Price (integer) — range: 33000 to 1918000, avg: 438939
  Sale Price (decimal) — range: 34000 to 1853000, avg: 447727
  Listing Status (text) — values: ['Active', 'Pending', 'Sold']


In [4]:
# Quick query example — how many listings per state (top 10)
result = dm.query(
    dataset="real_estate_listings",
    aggregation={"function": "count", "group_by": "State"}
)
# Sort by count descending
sorted_data = sorted(result['data'], key=lambda x: x.get('count', 0), reverse=True)[:10]
print("Top 10 States by Listing Count:")
for row in sorted_data:
    print(f"  {row['State']}: {row['count']} listings")

Top 10 States by Listing Count:
  Illinois: 116 listings
  Georgia: 113 listings
  Massachusetts: 108 listings
  Washington: 106 listings
  Florida: 98 listings
  Arizona: 96 listings
  Colorado: 94 listings
  New York: 91 listings
  Texas: 90 listings
  California: 88 listings


## 2. Defining and Testing Individual Tools

Each tool is self-contained with a name, description, parameter schema, and execute method.

In [5]:
from app.tools.base import ToolRegistry
from app.tools.query import QueryTool
from app.tools.insert import InsertTool
from app.tools.update import UpdateTool
from app.tools.delete import DeleteTool
from app.tools.schema_inspect import SchemaInspectTool
from app.tools.undo import UndoTool
from app.tools.list_changes import ListChangesTool
from app.data.validator import Validator

validator = Validator()
registry = ToolRegistry()

# Register all tools
registry.register(QueryTool(dm))
registry.register(InsertTool(dm, validator))
registry.register(UpdateTool(dm, validator))
registry.register(DeleteTool(dm))
registry.register(SchemaInspectTool(dm))
registry.register(UndoTool(dm))
registry.register(ListChangesTool(dm))

print(f"✅ Registered {len(registry.list_names())} tools: {registry.list_names()}")
print("\nTool Schemas (what the LLM sees):")
for schema in registry.get_schemas():
    print(f"  🔧 {schema['name']}: {schema['description'][:80]}...")

✅ Registered 7 tools: ['query_data', 'insert_data', 'update_data', 'delete_data', 'inspect_schema', 'undo_change', 'list_changes']

Tool Schemas (what the LLM sees):
  🔧 query_data: Query and search data from a dataset. Supports filtering by column values, sorti...
  🔧 insert_data: Insert new rows into a dataset. Validates data types, required fields, and value...
  🔧 update_data: Update existing rows in a dataset. Identifies rows using filters, validates new ...
  🔧 delete_data: Delete rows from a dataset. Identifies rows using filters, shows a preview of ro...
  🔧 inspect_schema: Inspect the structure and metadata of available datasets. Returns column names, ...
  🔧 undo_change: Undo a previous data mutation (insert, update, or delete). Can undo the most rec...
  🔧 list_changes: List recent data mutations (inserts, updates, deletes) from the change log. Show...


In [6]:
# Test QueryTool — simple filter
query_tool = registry.get("query_data")

result = query_tool.execute(
    dataset="real_estate_listings",
    filters=[{"column": "State", "operator": "eq", "value": "California"}],
    aggregation={"function": "count"}
)
print(f"Listings in California: {result.data['value']}")
print(f"Success: {result.success}")

Listings in California: 88
Success: True


In [7]:
# Test QueryTool — complex aggregation
result = query_tool.execute(
    dataset="real_estate_listings",
    filters=[
        {"column": "Property Type", "operator": "eq", "value": "House"},
        {"column": "Listing Status", "operator": "eq", "value": "Sold"}
    ],
    aggregation={"function": "avg", "column": "Sale Price", "group_by": "State"}
)
print("Average sale price of sold houses by state:")
sorted_data = sorted(result.data['data'], key=lambda x: x.get('avg_Sale Price', 0), reverse=True)[:5]
for row in sorted_data:
    print(f"  {row['State']}: ${row['avg_Sale Price']:,.0f}")

Average sale price of sold houses by state:
  Colorado: $1,006,421
  California: $954,706
  Washington: $857,400
  New York: $827,889
  Massachusetts: $809,500


In [8]:
# Test QueryTool — marketing campaigns, top by revenue
result = query_tool.execute(
    dataset="marketing_campaigns",
    sort_by="Revenue Generated",
    sort_order="desc",
    limit=5,
    columns=["Campaign ID", "Campaign Name", "Channel", "Revenue Generated"]
)
print("Top 5 Campaigns by Revenue:")
for row in result.data['data']:
    print(f"  {row['Campaign ID']}: {row['Campaign Name']} ({row['Channel']}) — ${row['Revenue Generated']:,.2f}")

Top 5 Campaigns by Revenue:
  CMP-8927: Summer Promo - Email 2025 Q1 (Email) — $503,194.13
  CMP-8682: Seasonal Collection - Email 2025 Q4 (Email) — $428,710.60
  CMP-8100: New Product Launch - Email 2025 Q3 (Email) — $424,886.78
  CMP-8696: Seasonal Collection - Email 2025 Q2 (Email) — $406,089.71
  CMP-8613: Brand Awareness - Email 2024 Q3 (Email) — $386,281.55


In [9]:
# Test Validator — valid data
valid_result = validator.validate_insert("real_estate_listings", [{
    "Listing ID": "LST-9999",
    "Property Type": "House",
    "City": "San Francisco",
    "State": "California",
    "Bedrooms": 3,
    "Bathrooms": 2.0,
    "Square Footage": 1500,
    "Year Built": 2020,
    "List Price": 850000,
    "Sale Price": 0,
    "Listing Status": "Active"
}])
print(f"Valid data: is_valid={valid_result.is_valid}, errors={valid_result.errors}")

# Test Validator — invalid data
invalid_result = validator.validate_insert("real_estate_listings", [{
    "Listing ID": "LST-9998",
    "Property Type": "Mansion",  # Invalid enum
    "Bedrooms": -1,  # Out of range
    "Year Built": 1600,  # Too old
}])
print(f"\nInvalid data: is_valid={invalid_result.is_valid}")
for err in invalid_result.errors:
    print(f"  ❌ {err}")

Valid data: is_valid=True, errors=[]

Invalid data: is_valid=False
  ❌ Row 1: Missing required field 'City'
  ❌ Row 1: Missing required field 'State'
  ❌ Row 1: 'Property Type' must be one of ['House', 'Condo', 'Apartment', 'Townhouse'], got 'Mansion'
  ❌ Row 1: 'Bedrooms' must be >= 0, got -1
  ❌ Row 1: 'Year Built' must be >= 1800, got 1600


In [10]:
# Test UpdateTool — preview without applying
update_tool = registry.get("update_data")

result = update_tool.execute(
    dataset="real_estate_listings",
    filters=[{"column": "Listing ID", "operator": "eq", "value": "LST-5001"}],
    updates={"List Price": 482000}
)
print(result.message)
print(f"\nRequires confirmation: {result.requires_confirmation}")

📝 About to update 1 row(s):

Before → After:

  Record 'LST-5001':
    List Price: 351000 → 482000

Proceed with update? (yes/no)

Requires confirmation: True


## 3. Building the Agent

The Agent combines the LLM provider, tool registry, and data manager into a
ReAct reasoning loop.

In [11]:
from app.llm.base import get_provider
from app.agent.core import Agent
from app.agent.session import SessionManager
from app.logging.logger import InteractionLogger

# Initialize LLM provider
llm = get_provider(
    provider_name=settings.llm_provider.value,
    api_key=settings.active_api_key,
    model=settings.active_model,
)
print(f"🤖 LLM Provider: {llm}")

# Build agent
agent = Agent(
    llm=llm,
    tool_registry=registry,
    data_manager=dm,
    logger=InteractionLogger(),
)

session_mgr = SessionManager()
print("✅ Agent ready!")

🤖 LLM Provider: GroqProvider(model='llama-3.3-70b-versatile')
✅ Agent ready!


## 4. Testing the Agent — End-to-End Scenarios

In [12]:
import asyncio

async def ask(question: str, session=None):
    """Helper to send a question to the agent and display the response."""
    if session is None:
        session = session_mgr.create_session()
    
    result = await agent.process_message(session, question)
    
    print(f"❓ {question}")
    print(f"💬 {result.response}")
    
    if result.reasoning_steps:
        print(f"\n📋 Reasoning ({len(result.reasoning_steps)} steps):")
        for step in result.reasoning_steps:
            if step.get('action'):
                print(f"   Step {step['step']}: Used tool '{step['action']}'")
    
    if result.tool_calls:
        print(f"\n🔧 Tools used: {[t['tool'] for t in result.tool_calls]}")
    
    if result.requires_confirmation:
        print(f"\n⚠️  Awaiting confirmation")
    
    print("-" * 60)
    return session, result

In [14]:
# Scenario 1: Simple count query
session, _ = await ask("How many listings are in California?")

01:53:47 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:53:48 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
01:53:49 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
01:53:52 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:53:52 | INFO    | agent | [7296dec0] Query: How many listings are in California?... | Tools: query_data | Latency: 5189ms


❓ How many listings are in California?
💬 There are 88 listings in California.

📋 Reasoning (2 steps):
   Step 1: Used tool 'query_data'

🔧 Tools used: ['query_data']
------------------------------------------------------------


In [15]:
# Scenario 2: Complex aggregation
_, _ = await ask("What is the average sale price by property type in Washington state?")

01:54:54 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:54:55 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:54:55 | INFO    | agent | [f6d38a83] Query: What is the average sale price by property type in Washington state?... | Tools: query_data | Latency: 2129ms


❓ What is the average sale price by property type in Washington state?
💬 The average sale price by property type in Washington state is as follows:
- Apartment: $261,615.38
- Condo: $394,375.00
- House: $883,032.26
- Townhouse: $612,500.00

📋 Reasoning (2 steps):
   Step 1: Used tool 'query_data'

🔧 Tools used: ['query_data']
------------------------------------------------------------


In [ ]:
# Scenario 3: Schema inspection
_, _ = await ask("What datasets do you have? Show me their structure.")

In [ ]:
# Scenario 4: Top-N query
_, _ = await ask("Show me the top 5 most expensive properties that have been sold")

In [ ]:
# Scenario 5: Marketing data query
_, _ = await ask("Which marketing channel generates the most revenue on average?")

In [ ]:
# Scenario 6: Multi-turn conversation
session, _ = await ask("How many houses are in Texas?")
# Follow-up using the same session
import time; time.sleep(3)  # Avoid rate limits
_, _ = await ask("What's the average price there?", session=session)

In [16]:
# Scenario 7: Update with preview + confirmation
import time; time.sleep(3)
session, result = await ask("Update the list price of LST-5002 to 750000")

if result.requires_confirmation:
    print("\n✅ Confirming the update...")
    time.sleep(3)
    _, confirm_result = await ask("yes", session=session)
    
    # Verify the change
    time.sleep(3)
    _, _ = await ask("What is the current list price of LST-5002?", session=session)

01:56:16 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:56:16 | INFO    | agent | [978e6d52] Query: Update the list price of LST-5002 to 750000... | Tools: update_data | Latency: 961ms


❓ Update the list price of LST-5002 to 750000
💬 📝 About to update 1 row(s):

Before → After:

  Record 'LST-5002':
    List Price: 709000 → 750000

Proceed with update? (yes/no)

📋 Reasoning (1 steps):
   Step 1: Used tool 'update_data'

🔧 Tools used: ['update_data']

⚠️  Awaiting confirmation
------------------------------------------------------------

✅ Confirming the update...


01:56:20 | INFO    | agent | [978e6d52] Query: yes... | Tools: none | Latency: 118ms


❓ yes
💬 ✅ Successfully updated 1 row(s). (Action ID: act_e599cc93 — use this to undo if needed)

📋 Reasoning (1 steps):
   Step 1: Used tool 'execute_update'
------------------------------------------------------------


01:56:23 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:56:24 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
01:56:25 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
01:56:28 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:56:28 | INFO    | agent | [978e6d52] Query: What is the current list price of LST-5002?... | Tools: query_data | Latency: 5522ms


❓ What is the current list price of LST-5002?
💬 The current list price of LST-5002 is $750,000.

📋 Reasoning (2 steps):
   Step 1: Used tool 'query_data'

🔧 Tools used: ['query_data']
------------------------------------------------------------


In [17]:
# Scenario 8: Undo the change
import time; time.sleep(3)
session, result = await ask("Undo the last change")

if result.requires_confirmation:
    print("\n✅ Confirming the undo...")
    time.sleep(3)
    _, _ = await ask("yes", session=session)

01:59:23 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:59:23 | INFO    | agent | [30ec376e] Query: Undo the last change... | Tools: undo_change | Latency: 674ms


❓ Undo the last change
💬 ⏪ Undo update (action: act_e599cc93) on real_estate_listings:

Reverting values:

  Record 'LST-5002':
    List Price: 750000 → 709000 (restoring original)

Proceed with undo? (yes/no)

📋 Reasoning (1 steps):
   Step 1: Used tool 'undo_change'

🔧 Tools used: ['undo_change']

⚠️  Awaiting confirmation
------------------------------------------------------------

✅ Confirming the undo...


01:59:26 | INFO    | agent | [30ec376e] Query: yes... | Tools: none | Latency: 139ms


❓ yes
💬 ✅ Successfully undone undo_update — 1 row(s) reverted.

📋 Reasoning (1 steps):
   Step 1: Used tool 'execute_undo'
------------------------------------------------------------


In [19]:
# Scenario 9: View change history
import time; time.sleep(3)
_, _ = await ask("Show me the recent change history")

02:00:34 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
02:00:36 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
02:00:38 | INFO    | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
02:00:38 | INFO    | agent | [ba468489] Query: Show me the recent change history... | Tools: none | Latency: 4139ms


❓ Show me the recent change history
💬 I encountered an error communicating with the LLM: Client error '429 Too Many Requests' for url 'https://api.groq.com/openai/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

📋 Reasoning (1 steps):
------------------------------------------------------------


## 5. Examining Structured Logs

Every interaction is logged as a JSON file in the `logs/` directory.

In [ ]:
from pathlib import Path
import json

logs_dir = Path('../logs')
log_files = sorted(logs_dir.glob('*.json'))

print(f"📋 Found {len(log_files)} interaction logs\n")

if log_files:
    # Show the latest log
    latest = log_files[-1]
    with open(latest) as f:
        log = json.load(f)
    
    print(f"Latest interaction:")
    print(f"  Query: {log['user_query']}")
    print(f"  Tools used: {[t['tool'] for t in log['tool_decisions']]}")
    print(f"  Response: {log['final_response'][:200]}")
    print(f"  Latency: {log['latency_ms']}ms")
    print(f"  Provider: {log['llm_provider']}")

## Summary

This notebook demonstrated:
1. **Data loading** — Excel files into pandas DataFrames
2. **Tool system** — 7 custom tools with validation and preview
3. **Agent reasoning** — ReAct loop with step-by-step reasoning
4. **Full CRUD** — Query, insert, update, delete with confirmation
5. **Undo mechanism** — Revert any past mutation
6. **Multi-turn** — Session memory for follow-up questions
7. **Structured logging** — Full traceability of every interaction